<img src="Bilder/ost_logo.png" width="240" align="right"/>
<div style="text-align: left"> <b> Machine Learning Projekt</b> <br> HS 2025 <br> <a href="mailto:ryan.mikus@ost.ch"> Ryan Mikus </a> und <a href="mailto:ilija.amato@ost.ch"> Ilija Amato </a> </div>

# 2 Datensammlung

## 2.1 Ziel der Datensammlung

Für die Entwicklung und Evaluation der geplanten Relative-Value-Trading-Strategie benötigen wir hochfrequente Kursdaten eines breiten, liquiden Aktienuniversums.  
In diesem Projekt arbeiten wir mit **1-Minuten-Intradaydaten** für alle Aktien des **S&P 500**. Diese Daten bilden die Grundlage für die anschließende explorative Datenanalyse (EDA) sowie für das spätere Feature Engineering und die Modellierung mit XGBoost.

Die Datensammlung gliedert sich in zwei Hauptschritte:

1. Auswahl und Speicherung des Aktienuniversums (S&P-500-Ticker)  
2. Abruf und Speicherung der Intraday-Kurszeitreihen über die (inoffizielle) Yahoo-Finance-API


In [7]:
import pandas as pd
import requests
import time
from datetime import datetime

## 2.2 Auswahl des Aktienuniversums (S&P-500-Ticker)

Als Aktienuniversum verwenden wir den **S&P 500**, einen der wichtigsten US-Aktienindizes.  
Anstatt die Ticker manuell zu pflegen, werden sie automatisiert von Wikipedia geladen. Hierzu wird die Seite  
[*List of S&P 500 companies*](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies) verwendet.

Mit `pandas.read_html` wird die erste Tabelle der Seite eingelesen, aus der die Spalte `Symbol` extrahiert und in eine Python-Liste umgewandelt wird.  
Da Yahoo Finance keine Punkte (`.`) in den Tickersymbolen akzeptiert (z. B. `BRK.B` → `BRK-B`), werden diese automatisch ersetzt.

Abschließend werden alle bereinigten Symbole in einer CSV-Datei gespeichert:

- **Datei:** `Daten/sp500_tickers.csv`  
- **Spalte:** `symbol`  
- Kein Index (für einfache Weiterverarbeitung)

Damit steht eine reproduzierbare und leicht aktualisierbare Liste der S&P-500-Ticker zur Verfügung.


In [8]:
# S&P-500-Ticker von Wikipedia holen
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
HEADERS = {"User-Agent": "Mozilla/5.0"}

resp = requests.get(WIKI_URL, headers=HEADERS)
resp.raise_for_status()

tables = pd.read_html(resp.text)
sp500 = tables[0]

# Symbol-Spalte
tickers = sp500["Symbol"].tolist()

# Yahoo mag keine Punkte – BRK.B → BRK-B
tickers = [t.replace(".", "-") for t in tickers]

print("Anzahl Ticker:", len(tickers))
print("Beispiele:", tickers[:10])

# In CSV speichern ===
pd.DataFrame({"symbol": tickers}).to_csv("Daten/sp500_tickers.csv", index=False)
print("Gespeichert als sp500_tickers.csv")


C:\Users\ilija\AppData\Local\Temp\ipykernel_32144\95533622.py:8: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text)


Anzahl Ticker: 503
Beispiele: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']
Gespeichert als sp500_tickers.csv


## 2.3 Abruf der Intraday-Kursdaten über die Yahoo-Finance-API

Für die eigentlichen Kursdaten nutzen wir **1-Minuten-Intradaydaten** von **Yahoo Finance**.  
Da die Python-Bibliothek `yfinance` in der Projektumgebung nicht einsatzfähig war, erfolgt der Abruf direkt über die inoffizielle JSON-API von Yahoo Finance  
(`https://query1.finance.yahoo.com/v8/finance/chart/{ticker}`).

Dieser Ansatz ist in der Community gut dokumentiert, auch wenn es sich nicht um eine offiziell unterstützte Schnittstelle handelt.  
Ein Beispiel für eine Beschreibung der API ist hier zu finden:  
[Guide to Yahoo Finance API (Scrapfly)](https://scrapfly.io/blog/posts/guide-to-yahoo-finance-api)

### 2.3.1 Parameterdefinition

Zu Beginn werden die zentralen Parameter festgelegt:

- `DATE` → Datum des Abrufs  
- `INTERVAL` → Zeitintervall (z. B. 1m)  
- `RANGE` → Zeitraum (z. B. 1d)  
- `OUTPUT_FILE` → Pfad für die Zieldatei

Die zuvor gespeicherten Ticker werden eingelesen, und für jeden Ticker werden über die Funktion `fetch_yahoo_chart(...)` die Kurszeitreihen abgerufen.  
Die Funktion extrahiert die Zeitstempel und Kursdaten aus der JSON-Struktur und gibt sie als `pandas.DataFrame` zurück.

Fehlerhafte oder leere Antworten werden erkannt und übersprungen, um den Download-Prozess robust zu halten.


In [9]:
# Einstellungen
DATE = "2025-11-07"       # gewünschtes Datum
INTERVAL = "1m"           # 1-Minute-Daten
RANGE = "1d"              # ein Tag
OUTPUT_FILE = f"Daten/sp500_1min_{DATE}.csv"

# Header wie Browser (damit Yahoo antwortet)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept": "application/json, text/javascript, */*; q=0.01",
}

# Ticker-Liste laden
tickers = pd.read_csv("Daten/sp500_tickers.csv")["symbol"].tolist()
print("Lade Daten für", len(tickers), "Ticker …")

# Funktion zum Laden eines Tickers
def fetch_yahoo_chart(ticker, interval="1m", range_="1d"):
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
    params = {"interval": interval, "range": range_, "region": "US", "lang": "en-US"}
    r = requests.get(url, headers=HEADERS, params=params, timeout=10)
    if r.status_code != 200:
        print(f"{ticker}: HTTP {r.status_code}")
        return pd.DataFrame()
    data = r.json().get("chart", {}).get("result")
    if not data:
        return pd.DataFrame()
    result = data[0]
    timestamps = result.get("timestamp")
    quotes = result.get("indicators", {}).get("quote", [{}])[0]
    if not timestamps or "close" not in quotes:
        return pd.DataFrame()
    df = pd.DataFrame({
        "symbol": ticker,
        "datetime": pd.to_datetime(timestamps, unit="s"),
        "open": quotes["open"],
        "high": quotes["high"],
        "low": quotes["low"],
        "close": quotes["close"],
        "volume": quotes["volume"],
    })
    return df

# Alle Ticker durchgehen
all_data = []
failed = []

Lade Daten für 503 Ticker …


## 2.4 Schleife über alle S&P-500-Ticker und Speicherung

Für jeden Ticker im S&P 500 wird die Funktion `fetch_yahoo_chart(...)` aufgerufen.  
Die Daten werden gesammelt, in einer Liste zusammengeführt und schließlich als CSV-Datei gespeichert.

- Erfolgreich geladene Ticker werden zu `all_data` hinzugefügt  
- Fehlgeschlagene Ticker landen in der Liste `failed`  
- Nach jedem Request wird mit `time.sleep(0.1)` eine kleine Pause eingelegt, um das API-Rate-Limit zu respektieren

Nach Abschluss der Schleife:

- Alle DataFrames werden mit `pd.concat` zu einem Gesamt-Dataset verbunden  
- Das Ergebnis wird gespeichert als  
  **`Daten/sp500_1min_2024-11-07.csv`**

Falls keine Daten geladen wurden, wird eine entsprechende Fehlermeldung ausgegeben.  
Eine Liste der nicht verarbeiteten Ticker dient zur schnellen Fehlerdiagnose.


In [10]:
for i, t in enumerate(tickers, start=1):
    df_t = fetch_yahoo_chart(t, INTERVAL, RANGE)
    if not df_t.empty:
        all_data.append(df_t)
        print(f"[{i}/{len(tickers)}] {t}: {len(df_t)} Zeilen")
    else:
        failed.append(t)
        print(f"[{i}/{len(tickers)}] {t}: keine Daten")
    time.sleep(0.1)  # sanftes Rate-Limit

# Zusammenführen & speichern
if all_data:
    df_all = pd.concat(all_data, ignore_index=True)
    df_all.to_csv(OUTPUT_FILE, index=False)
    print(f"\nGespeichert als {OUTPUT_FILE} ({len(df_all)} Zeilen)")
else:
    print("\nKeine Daten geladen.")

if failed:
    print("\nFehlgeschlagene Ticker:", failed[:20], "…")

[1/503] MMM: 391 Zeilen
[2/503] AOS: 391 Zeilen
[3/503] ABT: 391 Zeilen
[4/503] ABBV: 391 Zeilen
[5/503] ACN: 391 Zeilen
[6/503] ADBE: 391 Zeilen
[7/503] AMD: 391 Zeilen
[8/503] AES: 391 Zeilen
[9/503] AFL: 391 Zeilen
[10/503] A: 391 Zeilen
[11/503] APD: 391 Zeilen
[12/503] ABNB: 391 Zeilen
[13/503] AKAM: 391 Zeilen
[14/503] ALB: 391 Zeilen
[15/503] ARE: 391 Zeilen
[16/503] ALGN: 391 Zeilen
[17/503] ALLE: 391 Zeilen
[18/503] LNT: 391 Zeilen
[19/503] ALL: 391 Zeilen
[20/503] GOOGL: 391 Zeilen
[21/503] GOOG: 391 Zeilen
[22/503] MO: 391 Zeilen
[23/503] AMZN: 391 Zeilen
[24/503] AMCR: 391 Zeilen
[25/503] AEE: 391 Zeilen
[26/503] AEP: 391 Zeilen
[27/503] AXP: 391 Zeilen
[28/503] AIG: 391 Zeilen
[29/503] AMT: 391 Zeilen
[30/503] AWK: 391 Zeilen
[31/503] AMP: 391 Zeilen
[32/503] AME: 391 Zeilen
[33/503] AMGN: 391 Zeilen
[34/503] APH: 391 Zeilen
[35/503] ADI: 391 Zeilen
[36/503] AON: 391 Zeilen
[37/503] APA: 391 Zeilen
[38/503] APO: 391 Zeilen
[39/503] AAPL: 391 Zeilen
[40/503] AMAT: 391 Zeile

ReadTimeout: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Read timed out. (read timeout=10)

## 2.5 Einlesen und erste Plausibilitätsprüfung

Im letzten Schritt wird geprüft, ob die Datensammlung erfolgreich war.  
Dazu wird die erzeugte Datei `Daten/sp500_1min_2024-11-07.csv` wieder eingelesen und überprüft:

- Anzahl der Zeilen und Ticker  
- Zeitbereich der geladenen Daten (`datetime.min()` → `datetime.max()`)  
- Kontrolle, ob `datetime` korrekt als Datumsobjekt erkannt wurde  

Diese Prüfung bestätigt, dass die Daten konsistent und vollständig sind.  
Die so gewonnenen 1-Minuten-Kursdaten bilden die Grundlage für die anschließende **explorative Datenanalyse**, die im nächsten Kapitel behandelt wird:  
[03_Explorative_Datenanalyse](03_Explorative_Datenanalyse.html)


In [ ]:
DATE = "2025-11-07"  # gleiches Datum wie beim Download
FILE = f"Daten/sp500_1min_{DATE}.csv"

# === CSV einlesen ===
df = pd.read_csv(FILE, parse_dates=["datetime"])

print("Datei erfolgreich geladen:", FILE)
print("Anzahl Zeilen:", len(df))
print("Anzahl Ticker:", df['symbol'].nunique())
print("Zeitbereich:", df['datetime'].min(), "→", df['datetime'].max())

NameError: name 'pd' is not defined

---

<div style="display: flex; justify-content: space-between; margin-top: 30px;">
  <a href="01_Einleitung.html">
    <button style="background-color:#007acc;color:white;border:none;padding:8px 12px;border-radius:6px;cursor:pointer;">
      Zurück
    </button>
  </a>
  <a href="03_Explorative_Datenanalyse.html">
    <button style="background-color:#007acc;color:white;border:none;padding:8px 12px;border-radius:6px;cursor:pointer;">
      Weiter
    </button>
  </a>
</div>
